# Pandas Module: Advanced Column Processing and Feature Engineering

This module focuses on a common real-world task: **creating, updating, transforming, and deleting columns based on other columns**.

You will practice:

- Creating new columns from one or more existing columns
- Updating values using conditions
- Writing custom functions for row-level logic
- Using `apply()` and `lambda`
- Using vectorized alternatives such as `np.select()` and `np.where()`
- Creating group-based columns with `groupby().transform()`
- Creating dummy variables for categorical columns
- Dropping columns safely
- Building a clean feature-engineering workflow

The dataset is a synthetic retail transaction dataset designed for teaching.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## 1. Load the data

In [3]:
transactions = pd.read_csv("transactions_advanced.csv")
customers = pd.read_csv("customers_advanced.csv")

transactions.head()

,order_id,customer_id,order_date,country,channel,category,segment,payment_method,ship_method,quantity,unit_price,discount_rate,shipping_fee,status,satisfaction_score
0,ORD10228,C1013,2025-05-18,UK,Marketplace,Toys,Consumer,Credit Card,Express,5,60.26,NaN,0.00,Returned,4.0
1,ORD10194,C1022,2025-04-11,USA,Partner,Clothing,Consumer,Gift Card,Standard,1,67.94,0.15,12.99,Completed,4.0
2,ORD10088,C1033,2025-05-30,France,Marketplace,Toys,Consumer,Credit Card,Standard,3,62.34,0.10,12.99,Completed,3.0
3,ORD10095,C1034,2025-03-11,USA,Retail,Books,Enterprise,Credit Card,Standard,1,56.29,0.00,0.00,Cancelled,5.0
4,ORD10214,C1032,2025-06-03,UK,Online,Home,Consumer,Gift Card,Standard,7,102.61,0.00,4.99,Completed,4.0


In [ ]:
transactions.info()

## 2. Start with a copy

When transforming data, it is safer to keep the raw data unchanged and work on a copy.

In [5]:
df = transactions.copy()
df.head()

,order_id,customer_id,order_date,country,channel,category,segment,payment_method,ship_method,quantity,unit_price,discount_rate,shipping_fee,status,satisfaction_score
0,ORD10228,C1013,2025-05-18,UK,Marketplace,Toys,Consumer,Credit Card,Express,5,60.26,NaN,0.00,Returned,4.0
1,ORD10194,C1022,2025-04-11,USA,Partner,Clothing,Consumer,Gift Card,Standard,1,67.94,0.15,12.99,Completed,4.0
2,ORD10088,C1033,2025-05-30,France,Marketplace,Toys,Consumer,Credit Card,Standard,3,62.34,0.10,12.99,Completed,3.0
3,ORD10095,C1034,2025-03-11,USA,Retail,Books,Enterprise,Credit Card,Standard,1,56.29,0.00,0.00,Cancelled,5.0
4,ORD10214,C1032,2025-06-03,UK,Online,Home,Consumer,Gift Card,Standard,7,102.61,0.00,4.99,Completed,4.0


## 3. Create simple calculated columns

Many new columns are created directly from existing columns.

Here we create:

- `gross_sales = quantity * unit_price`
- `discount_amount = gross_sales * discount_rate`
- `net_sales = gross_sales - discount_amount + shipping_fee`

In [6]:
df["gross_sales"] = df["quantity"] * df["unit_price"]
df["discount_rate_clean"] = df["discount_rate"].fillna(0)
df["discount_amount"] = df["gross_sales"] * df["discount_rate_clean"]
df["net_sales"] = df["gross_sales"] - df["discount_amount"] + df["shipping_fee"]

df[["quantity", "unit_price", "discount_rate", "shipping_fee", "gross_sales", "discount_amount", "net_sales"]].head(10)

,quantity,unit_price,discount_rate,shipping_fee,gross_sales,discount_amount,net_sales
0,5,60.26,NaN,0.00,301.30,0.000,301.300
1,1,67.94,0.15,12.99,67.94,10.191,70.739
2,3,62.34,0.10,12.99,187.02,18.702,181.308
3,1,56.29,0.00,0.00,56.29,0.000,56.290
4,7,102.61,0.00,4.99,718.27,0.000,723.260
5,7,54.79,0.10,7.99,383.53,38.353,353.167
6,1,73.81,0.10,0.00,73.81,7.381,66.429
7,5,106.84,0.00,4.99,534.20,0.000,539.190
8,4,176.40,0.00,19.99,705.60,0.000,725.590
9,4,42.92,0.10,4.99,171.68,17.168,159.502


### Teaching note: why create `discount_rate_clean`?

The original `discount_rate` has missing values. Instead of overwriting it immediately, we create a clean version first. This makes the transformation easier to audit.

## 4. Updating column values with conditions

Use `.loc[row_condition, column_name] = new_value`.

Example: if an order is returned or cancelled, set `net_sales` to 0.

In [8]:
df["status"].isin(["Returned", "Cancelled"])

0       True
1      False
2      False
3       True
4      False
       ...  
235    False
236    False
237     True
238     True
239     True
Name: status, Length: 240, dtype: bool

In [9]:
df.loc[df["status"].isin(["Returned", "Cancelled"]), "net_sales"] = 0

df[["status", "quantity", "unit_price", "net_sales"]].head(2)

,status,quantity,unit_price,net_sales
0,Returned,5,60.26,0.000
1,Completed,1,67.94,70.739


### More detail: understanding `.loc`

`.loc` is one of the most important pandas tools for beginner students.

The basic pattern is:

```python
df.loc[row_condition, column_name] = new_value
```

Think of it as:

> Find the rows that satisfy a condition, then update a specific column for those rows.

This is safer and clearer than trying to update a filtered DataFrame directly.


In [10]:
# Create a small practice copy so we do not accidentally change the main df
loc_demo = transactions.copy()

# First, create columns needed for practice
loc_demo["discount_rate_clean"] = loc_demo["discount_rate"].fillna(0)
loc_demo["gross_sales"] = loc_demo["quantity"] * loc_demo["unit_price"]
loc_demo["net_sales"] = (
    loc_demo["gross_sales"] * (1 - loc_demo["discount_rate_clean"])
    + loc_demo["shipping_fee"]
)

loc_demo[["order_id", "status", "country", "quantity", "unit_price", "net_sales"]].head(2)


,order_id,status,country,quantity,unit_price,net_sales
0,ORD10228,Returned,UK,5,60.26,301.300
1,ORD10194,Completed,USA,1,67.94,70.739


#### Example 1: View rows with `.loc`

Before updating values, it is a good habit to first **view the rows** you are about to change.


In [11]:
# Boolean condition: status is Returned or Cancelled
problem_order_condition = loc_demo["status"].isin(["Returned", "Cancelled"])

# View selected rows and selected columns
loc_demo.loc[
    problem_order_condition,
    ["order_id", "status", "net_sales"]
].head(2)


,order_id,status,net_sales
0,ORD10228,Returned,301.30
3,ORD10095,Cancelled,56.29


#### Example 2: Update one column with `.loc`

Now we set `net_sales` to 0 for returned or cancelled orders.


In [ ]:
# Update net_sales only for returned or cancelled orders
loc_demo.loc[
    problem_order_condition,
    "net_sales"
] = 0

# Check the result
loc_demo.loc[
    problem_order_condition,
    ["order_id", "status", "net_sales"]
].head(10)


#### Example 3: Create a new column, then update selected rows

A common beginner-friendly pattern is:

1. Create a default value for all rows
2. Use `.loc` to change values only for selected rows


In [12]:
# Step 1: Give every row a default label
loc_demo["order_status_group"] = "Completed Order"

# Step 2: Change the label for returned or cancelled orders
loc_demo.loc[
    loc_demo["status"].isin(["Returned", "Cancelled"]),
    "order_status_group"
] = "Problem Order"

# Step 3: Inspect the result
loc_demo[["order_id", "status", "order_status_group"]].head(2)


,order_id,status,order_status_group
0,ORD10228,Returned,Problem Order
1,ORD10194,Completed,Completed Order


#### Example 4: Use multiple conditions with `.loc`

When combining conditions:

- Use `&` for **AND**
- Use `|` for **OR**
- Put each condition inside parentheses

Example:

```python
(df["country"] == "US") & (df["net_sales"] >= 300)
```


In [ ]:
# Create a VIP flag using multiple conditions
# VIP order = US order AND net_sales is at least 300

loc_demo["vip_order"] = 0

loc_demo.loc[
    (loc_demo["country"] == "US") & (loc_demo["net_sales"] >= 300),
    "vip_order"
] = 1

loc_demo[["order_id", "country", "net_sales", "vip_order"]].head(15)


#### Example 5: Update multiple columns at once

`.loc` can update more than one column.

Example: for cancelled orders, set:

- `net_sales` to 0
- `order_status_group` to `"Problem Order"`


In [ ]:
# Update two columns for cancelled orders
loc_demo.loc[
    loc_demo["status"] == "Cancelled",
    ["net_sales", "order_status_group"]
] = [0, "Problem Order"]

loc_demo.loc[
    loc_demo["status"] == "Cancelled",
    ["order_id", "status", "net_sales", "order_status_group"]
].head(10)


#### Example 6: Use `.loc` to fix missing values in one column

This is useful when only some rows need to be cleaned.


In [ ]:
# Count missing values before update
print("Missing country values before:", loc_demo["country"].isna().sum())

# Replace missing country values with "Unknown"
loc_demo.loc[
    loc_demo["country"].isna(),
    "country"
] = "Unknown"

# Count missing values after update
print("Missing country values after:", loc_demo["country"].isna().sum())


#### Common beginner mistake: chained assignment

Avoid this pattern:

```python
df[df["status"] == "Cancelled"]["net_sales"] = 0
```

It may not update the original DataFrame correctly.

Use `.loc` instead:

```python
df.loc[df["status"] == "Cancelled", "net_sales"] = 0
```


In [ ]:
# Correct version: update the original DataFrame with .loc
loc_demo.loc[
    loc_demo["status"] == "Cancelled",
    "net_sales"
] = 0


### Mini-check: `.loc` syntax

Ask students to read this line out loud:

```python
df.loc[df["net_sales"] > 300, "sales_level"] = "High"
```

Plain English:

> For rows where net_sales is greater than 300, set the sales_level column to High.


## 5. Create a flag column

A flag column usually contains 0/1 or True/False values.

Example: create `is_return_or_cancelled`.

In [13]:
df["is_return_or_cancelled"] = df["status"].isin(["Returned", "Cancelled"]).astype(int)

df[["status", "is_return_or_cancelled"]].head(10)

,status,is_return_or_cancelled
0,Returned,1
1,Completed,0
2,Completed,0
3,Cancelled,1
4,Completed,0
5,Completed,0
6,Completed,0
7,Completed,0
8,Completed,0
9,Cancelled,1


## 6. Create columns based on multiple conditions with `np.select`

`np.select()` is good when you have several conditions.

Example: create an `order_size` column:

- Large: net sales >= 300
- Medium: net sales >= 100 and < 300
- Small: net sales > 0 and < 100
- No Revenue: net sales == 0

In [15]:
def get_order_size(net_sales):
    if net_sales >= 300:
        return "Large"
    elif 100 <= net_sales < 300:
        return "Medium"
    elif 0 < net_sales < 100:
        return "Small"
    elif net_sales == 0:
        return "No Revenue"
    else:
        return "Unknown"

df["order_size"] = df["net_sales"].apply(get_order_size)

In [16]:
conditions = [
    df["net_sales"] >= 300,
    df["net_sales"].between(100, 299.99),
    df["net_sales"].between(0.01, 99.99),
    df["net_sales"] == 0
]

choices = ["Large", "Medium", "Small", "No Revenue"]

df["order_size"] = np.select(conditions, choices, default="Unknown")

df[["net_sales", "order_size"]].head(5)

,net_sales,order_size
0,0.000,No Revenue
1,70.739,Small
2,181.308,Medium
3,0.000,No Revenue
4,723.260,Large


## 7. Use a custom function for row-level logic

Sometimes business rules are easier to read as a function.

Example: create `risk_level` based on several columns:

- High Risk: returned/cancelled order, or wire payment with high sales
- Medium Risk: high discount or missing payment method
- Low Risk: otherwise

In [ ]:
def assign_risk_level(row):
    if row["status"] in ["Returned", "Cancelled"]:
        return "High Risk"
    elif row["payment_method"] == "Wire" and row["net_sales"] > 250:
        return "High Risk"
    elif row["discount_rate_clean"] >= 0.20:
        return "Medium Risk"
    elif pd.isna(row["payment_method"]):
        return "Medium Risk"
    else:
        return "Low Risk"

df["risk_level"] = df.apply(assign_risk_level, axis=1)

df[["status", "payment_method", "discount_rate_clean", "net_sales", "risk_level"]].head(15)

### Important: what does `axis=1` mean?

`axis=1` means the function receives one **row** at a time.

Without `axis=1`, pandas applies the function column-by-column, which is not what we want here.

## 8. Use `lambda` for short transformations

A `lambda` function is a short one-line function.

Example: create a simplified region column from country.

In [ ]:
df["region"] = df["country"].apply(
    lambda x: "North America" if x in ["USA", "Canada"] else "International"
)

df[["country", "region"]].head(15)

The marketing team wants a standardized 3-letter uppercase country code for a new reporting dashboard

In [19]:
# Using a lambda function within .apply()
df['country_code'] = df['country'].apply(
    # lambda x: str(x)[:3].upper() if pd.notnull(x) else "UNK"
    lambda x: str(x)[:3].upper()
)

print(df[['country', 'country_code']].head())

  country country_code
0      UK           UK
1     USA          USA
2  France          FRA
3     USA          USA
4      UK           UK


In [18]:
# 1. Define the logic as a reusable function
def format_country_code(country_name):
    # Standardize to string, take 3 chars, and uppercase
    return str(country_name)[:3].upper()

# 2. Use .apply() to pass every element of the column through this function
df['country_code'] = df['country'].apply(format_country_code)

print(df[['country', 'country_code']].head())

  country country_code
0      UK           UK
1     USA          USA
2  France          FRA
3     USA          USA
4      UK           UK


In [23]:
# apply() with multiple columns
def cal_total_price(row):
    return row['quantity'] * row['unit_price']
df['total_price'] = df.apply(cal_total_price, axis=1)

print(df[['order_id', 'quantity', 'unit_price', 'total_price']].head())

   order_id  quantity  unit_price  total_price
0  ORD10228         5       60.26       301.30
1  ORD10194         1       67.94        67.94
2  ORD10088         3       62.34       187.02
3  ORD10095         1       56.29        56.29
4  ORD10214         7      102.61       718.27


In [24]:
# One-liner using lambda
df['total_price'] = df.apply(
    lambda row: (row['quantity'] * row['unit_price']), axis=1
)

print(df[['order_id', 'quantity', 'unit_price', 'total_price']].head())

   order_id  quantity  unit_price  total_price
0  ORD10228         5       60.26       301.30
1  ORD10194         1       67.94        67.94
2  ORD10088         3       62.34       187.02
3  ORD10095         1       56.29        56.29
4  ORD10214         7      102.61       718.27


### Caution

Use `lambda` for simple logic. If the logic becomes long, write a named function instead.

## 9. Prefer vectorized operations when possible

`apply()` is flexible but can be slower. For many tasks, vectorized pandas operations are cleaner and faster.

Example: create `region_v2` using `np.where()`.

In [ ]:
df["region_v2"] = np.where(
    df["country"].isin(["USA", "Canada"]),
    "North America",
    "International"
)

df[["country", "region", "region_v2"]].head(10)

In [25]:
# 1. Define the logic
def get_region(country):
    north_america = ["USA", "Canada"]
    if country in north_america:
        return "North America"
    else:
        return "International"

# 2. Apply it to the 'country' column
df['region_v2'] = df['country'].apply(get_region)

print(df[['country', 'region_v2']].head(10))

     country      region_v2
0         UK  International
1        USA  North America
2     France  International
3        USA  North America
4         UK  International
5         UK  International
6        USA  North America
7     Brazil  International
8        USA  North America
9  Australia  International


In [26]:
# The lambda version uses a Python 'ternary operator' (if-else in one line)
df['region_v2'] = df['country'].apply(
    lambda x: "North America" if x in ["USA", "Canada"] else "International"
)

print(df[['country', 'region_v2']].head(10))

     country      region_v2
0         UK  International
1        USA  North America
2     France  International
3        USA  North America
4         UK  International
5         UK  International
6        USA  North America
7     Brazil  International
8        USA  North America
9  Australia  International


## 10. Processing a group of columns

Sometimes we need to process several columns together.

Example: compute the number of missing values across selected customer/order fields.

In [27]:
important_cols = ["country", "channel", "payment_method", "unit_price", "discount_rate"]

df["missing_count"] = df[important_cols].isna().sum(axis=1)

df[important_cols + ["missing_count"]].head(2)

,country,channel,payment_method,unit_price,discount_rate,missing_count
0,UK,Marketplace,Credit Card,60.26,NaN,1
1,USA,Partner,Gift Card,67.94,0.15,0


## 11. Row-level score based on multiple columns

Create a `data_quality_score`:

- Start at 100
- subtract 20 for each missing important field
- subtract 20 if `net_sales` is negative
- subtract 10 if satisfaction score is missing

In [ ]:
df["data_quality_score"] = 100
df["data_quality_score"] = df["data_quality_score"] - 20 * df["missing_count"]
df.loc[df["net_sales"] < 0, "data_quality_score"] -= 20
df.loc[df["satisfaction_score"].isna(), "data_quality_score"] -= 10

df["data_quality_score"] = df["data_quality_score"].clip(lower=0, upper=100)

df[important_cols + ["net_sales", "satisfaction_score", "data_quality_score"]].head(15)

## 12. Group-based columns with `groupby().transform()`

Sometimes a column needs to compare each row with its group.

Example: create each order's share of total category sales.

In [33]:
df.groupby("category")["net_sales"].transform("sum")

0      6097.5440
1      6771.4290
2      6097.5440
3      5797.9525
4      7453.0690
         ...    
235    6097.5440
236    7453.0690
237    7453.0690
238    7453.0690
239    6174.4000
Name: net_sales, Length: 240, dtype: float64

In [ ]:
category_total_sales = df.groupby("category")["net_sales"].transform("sum")

df["category_sales_share"] = df["net_sales"] / category_total_sales

df[["category", "net_sales", "category_sales_share"]].head(15)

### Why `transform()` instead of `agg()`?

- `agg()` returns one row per group.
- `transform()` returns one value per original row.

That makes `transform()` perfect for creating new columns.

## 13. Create group-level benchmarks

Example: compare each order to the average net sales in its category.

In [ ]:
df["category_avg_net_sales"] = df.groupby("category")["net_sales"].transform("mean")

df["above_category_average"] = df["net_sales"] > df["category_avg_net_sales"]

df[["category", "net_sales", "category_avg_net_sales", "above_category_average"]].head(15)

## 14. Rank within a group

Example: rank orders within each country by net sales.

In [ ]:
df["rank_within_country"] = df.groupby("country")["net_sales"].rank(method="dense", ascending=False)

df[["country", "order_id", "net_sales", "rank_within_country"]].sort_values(["country", "rank_within_country"]).head(20)

Smart Data Imputation (Filling NaNs with Group Medians)

In [ ]:
# 1. Calculate the median score for each country and map it to every row
country_median_scores = df.groupby('country')['satisfaction_score'].transform('median')

# 2. Use fillna to apply these specific medians only where data is missing
# when you pass a Series (like country_median_scores) into .fillna(), 
# Pandas doesn't just fill values randomly. It matches them by their Index.

df['satisfaction_score'] = df['satisfaction_score'].fillna(country_median_scores)

# Note: Any remaining NaNs would be for countries that have NO scores at all.
print(df[['country', 'satisfaction_score']].head(2))

  country  satisfaction_score
0      UK                 4.0
1     USA                 4.0


## 15. Merge customer data and create columns from both tables

In real projects, the columns you need are often spread across multiple tables.

In [ ]:
df = df.merge(customers, on="customer_id", how="left")

df[["customer_id", "loyalty_tier", "marketing_opt_in", "signup_date"]].head()

Now create a column using both transaction and customer information.

Example: `priority_customer_order` is 1 if:

- loyalty tier is Gold or Platinum, and
- net sales is above 200

In [ ]:
df["priority_customer_order"] = (
    df["loyalty_tier"].isin(["Gold", "Platinum"]) & 
    (df["net_sales"] > 200)
).astype(int)

df[["customer_id", "loyalty_tier", "net_sales", "priority_customer_order"]].head(15)

## 16. Create dummy variables

Dummy variables convert categories into 0/1 columns. This is useful for machine learning and regression models.

Example: create dummy columns for `channel`.

In [ ]:
channel_dummies = pd.get_dummies(df["channel"], prefix="channel", dummy_na=True)

channel_dummies.head()

In [ ]:
df = pd.concat([df, channel_dummies], axis=1)

df.filter(regex="^channel_").head()

### Avoid the dummy variable trap

For regression models, you may want to drop one category:

```python
pd.get_dummies(df["channel"], prefix="channel", drop_first=True)
```

For tree-based machine learning models, keeping all dummy columns is usually fine.

## 17. Update values using `replace()` and `map()`

`replace()` is useful for changing values directly.

`map()` is useful for converting categories into labels or scores.

In [ ]:
df["status_clean"] = df["status"].replace({
    "Completed": "complete",
    "Returned": "problem",
    "Cancelled": "problem"
})

tier_score_map = {
    "Bronze": 1,
    "Silver": 2,
    "Gold": 3,
    "Platinum": 4
}

df["loyalty_score"] = df["loyalty_tier"].map(tier_score_map)

df[["status", "status_clean", "loyalty_tier", "loyalty_score"]].head(15)

## 18. Delete columns safely

Use `.drop(columns=[...])`.

Set `errors="ignore"` if the column may or may not exist.

In [ ]:
df = df.drop(columns=["region_v2"], errors="ignore")

"region_v2" in df.columns

## 19. Build a reusable cleaning function

A good workflow packages repeated transformations into a function.

In [ ]:
def prepare_transactions(transactions, customers):
    data = transactions.copy()

    data["discount_rate_clean"] = data["discount_rate"].fillna(0)
    data["gross_sales"] = data["quantity"] * data["unit_price"]
    data["discount_amount"] = data["gross_sales"] * data["discount_rate_clean"]
    data["net_sales"] = data["gross_sales"] - data["discount_amount"] + data["shipping_fee"]
    data.loc[data["status"].isin(["Returned", "Cancelled"]), "net_sales"] = 0

    data["is_return_or_cancelled"] = data["status"].isin(["Returned", "Cancelled"]).astype(int)

    conditions = [
        data["net_sales"] >= 300,
        data["net_sales"].between(100, 299.99),
        data["net_sales"].between(0.01, 99.99),
        data["net_sales"] == 0
    ]
    choices = ["Large", "Medium", "Small", "No Revenue"]
    data["order_size"] = np.select(conditions, choices, default="Unknown")

    data["region"] = np.where(
        data["country"].isin(["USA", "Canada"]),
        "North America",
        "International"
    )

    important_cols = ["country", "channel", "payment_method", "unit_price", "discount_rate"]
    data["missing_count"] = data[important_cols].isna().sum(axis=1)
    data["data_quality_score"] = 100 - 20 * data["missing_count"]
    data.loc[data["net_sales"] < 0, "data_quality_score"] -= 20
    data.loc[data["satisfaction_score"].isna(), "data_quality_score"] -= 10
    data["data_quality_score"] = data["data_quality_score"].clip(lower=0, upper=100)

    data["category_total_sales"] = data.groupby("category")["net_sales"].transform("sum")
    data["category_sales_share"] = data["net_sales"] / data["category_total_sales"]
    data["category_avg_net_sales"] = data.groupby("category")["net_sales"].transform("mean")
    data["above_category_average"] = data["net_sales"] > data["category_avg_net_sales"]

    data = data.merge(customers, on="customer_id", how="left")
    data["priority_customer_order"] = (
        data["loyalty_tier"].isin(["Gold", "Platinum"]) & 
        (data["net_sales"] > 200)
    ).astype(int)

    return data

prepared = prepare_transactions(transactions, customers)
prepared.head()

## 20. Check your results

A useful final step is to inspect the columns you created.

In [ ]:
created_cols = [
    "gross_sales", "discount_rate_clean", "discount_amount", "net_sales",
    "is_return_or_cancelled", "order_size", "region", "missing_count",
    "data_quality_score", "category_sales_share", "above_category_average",
    "priority_customer_order"
]

prepared[created_cols].head(10)

In [ ]:
prepared.groupby(["country", "channel"])["net_sales"].sum().sort_values(ascending=False).head(20)

## 21. Cleaner multi-line style for long pandas chains

For readability, wrap the expression in parentheses and put each operation on its own line.

In [ ]:
(
    prepared
    .groupby(["country", "channel"])["net_sales"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

# Practice Exercises

Use `transactions_advanced.csv` and `customers_advanced.csv`.

## Exercise 1
Create a new column `sales_before_shipping`.

Formula:

`quantity * unit_price * (1 - discount_rate_clean)`

## Exercise 2
Create a column `shipping_category`:

- Free Shipping: shipping fee is 0
- Low Shipping: shipping fee <= 7.99
- High Shipping: shipping fee > 7.99

## Exercise 3
Create a custom function to classify orders:

- "Problem Order" if status is Returned or Cancelled
- "High Value" if net sales >= 300
- "Normal" otherwise

Apply it row-by-row.

## Exercise 4
Create a column showing each order's share of total country sales.

## Exercise 5
Create dummy variables for `payment_method`.

## Exercise 6
Update missing `country` values to `"Unknown"`.

## Exercise 7
Drop the temporary column `discount_rate_clean`.

## Exercise 8
Create a reusable function called `make_features()` that performs Exercises 1–6.

# Suggested Solutions

In [ ]:
ex = transactions.copy()

ex["discount_rate_clean"] = ex["discount_rate"].fillna(0)
ex["sales_before_shipping"] = ex["quantity"] * ex["unit_price"] * (1 - ex["discount_rate_clean"])

ex[["quantity", "unit_price", "discount_rate", "sales_before_shipping"]].head()

In [ ]:
conditions = [
    ex["shipping_fee"] == 0,
    ex["shipping_fee"] <= 7.99,
    ex["shipping_fee"] > 7.99
]

choices = ["Free Shipping", "Low Shipping", "High Shipping"]

ex["shipping_category"] = np.select(conditions, choices, default="Unknown")

ex[["shipping_fee", "shipping_category"]].head(10)

In [ ]:
ex["gross_sales"] = ex["quantity"] * ex["unit_price"]
ex["net_sales"] = ex["gross_sales"] * (1 - ex["discount_rate_clean"]) + ex["shipping_fee"]
ex.loc[ex["status"].isin(["Returned", "Cancelled"]), "net_sales"] = 0

def classify_order(row):
    if row["status"] in ["Returned", "Cancelled"]:
        return "Problem Order"
    elif row["net_sales"] >= 300:
        return "High Value"
    else:
        return "Normal"

ex["order_class"] = ex.apply(classify_order, axis=1)

ex[["status", "net_sales", "order_class"]].head(15)

In [ ]:
ex["country_total_sales"] = ex.groupby("country")["net_sales"].transform("sum")
ex["country_sales_share"] = ex["net_sales"] / ex["country_total_sales"]

ex[["country", "net_sales", "country_sales_share"]].head(15)

In [ ]:
payment_dummies = pd.get_dummies(ex["payment_method"], prefix="payment", dummy_na=True)
ex = pd.concat([ex, payment_dummies], axis=1)

ex.filter(regex="^payment_").head()

In [ ]:
ex["country"] = ex["country"].fillna("Unknown")

ex["country"].isna().sum()

In [ ]:
ex = ex.drop(columns=["discount_rate_clean"], errors="ignore")

"discount_rate_clean" in ex.columns

In [ ]:
def make_features(data):
    out = data.copy()
    out["discount_rate_clean"] = out["discount_rate"].fillna(0)
    out["sales_before_shipping"] = out["quantity"] * out["unit_price"] * (1 - out["discount_rate_clean"])

    conditions = [
        out["shipping_fee"] == 0,
        out["shipping_fee"] <= 7.99,
        out["shipping_fee"] > 7.99
    ]
    choices = ["Free Shipping", "Low Shipping", "High Shipping"]
    out["shipping_category"] = np.select(conditions, choices, default="Unknown")

    out["gross_sales"] = out["quantity"] * out["unit_price"]
    out["net_sales"] = out["gross_sales"] * (1 - out["discount_rate_clean"]) + out["shipping_fee"]
    out.loc[out["status"].isin(["Returned", "Cancelled"]), "net_sales"] = 0

    def classify_order(row):
        if row["status"] in ["Returned", "Cancelled"]:
            return "Problem Order"
        elif row["net_sales"] >= 300:
            return "High Value"
        else:
            return "Normal"

    out["order_class"] = out.apply(classify_order, axis=1)
    out["country"] = out["country"].fillna("Unknown")
    out["country_total_sales"] = out.groupby("country")["net_sales"].transform("sum")
    out["country_sales_share"] = out["net_sales"] / out["country_total_sales"]

    return out

features = make_features(transactions)
features.head()

# Additional Classroom Practice: `.loc` and Column Updates

These exercises are designed for beginner students. Each one practices a small idea.

Use a fresh copy of the transactions data.


In [ ]:
practice = transactions.copy()

# Create basic columns for the exercises
practice["discount_rate_clean"] = practice["discount_rate"].fillna(0)
practice["gross_sales"] = practice["quantity"] * practice["unit_price"]
practice["net_sales"] = (
    practice["gross_sales"] * (1 - practice["discount_rate_clean"])
    + practice["shipping_fee"]
)

practice.head()


## `.loc` Exercise 1

Create a new column called `review_needed`.

Set the default value to `"No"`.

Then use `.loc` to change it to `"Yes"` when the order status is `"Returned"` or `"Cancelled"`.


## `.loc` Exercise 2

Create a new column called `high_shipping_flag`.

Set it to `1` when `shipping_fee` is greater than 10.

Otherwise, it should be `0`.


## `.loc` Exercise 3

Create a column called `discount_group`.

Use this logic:

- `"No Discount"` if `discount_rate_clean` is 0
- `"Small Discount"` if discount is greater than 0 and less than or equal to 0.10
- `"Large Discount"` if discount is greater than 0.10


## `.loc` Exercise 4

For returned or cancelled orders, update `net_sales` to 0.

Then display only these columns:

`order_id`, `status`, `net_sales`


## `.loc` Exercise 5

Create a column called `important_order`.

Set it to `"Yes"` if:

- `net_sales` is at least 300, AND
- `channel` is `"Online"`

Otherwise, set it to `"No"`.


## `.loc` Exercise 6

Create a column called `shipping_fee_adjusted`.

Start by copying the original `shipping_fee`.

Then set `shipping_fee_adjusted` to 0 when the order is a returned or cancelled order.


## `.loc` Exercise 7

Fix missing values:

- Replace missing `country` values with `"Unknown"`
- Replace missing `customer_segment` values with `"Unassigned"`

Then check whether missing values remain in those two columns.


## `.loc` Exercise 8

Create a new column called `sales_note`.

Use this logic:

- `"High value completed order"` if `net_sales >= 300` and status is `"Completed"`
- `"Problem order"` if status is `"Returned"` or `"Cancelled"`
- `"Regular order"` otherwise


## `.loc` Exercise 9

Update multiple columns at the same time.

For cancelled orders:

- Set `net_sales` to 0
- Set `sales_note` to `"Cancelled order"`


## `.loc` Exercise 10

Create dummy columns for `channel`.

Then combine them with the original DataFrame.

Hint: use `pd.get_dummies()`.


# Additional Classroom Practice: Suggested Solutions


In [ ]:
# Solution setup
practice = transactions.copy()

practice["discount_rate_clean"] = practice["discount_rate"].fillna(0)
practice["gross_sales"] = practice["quantity"] * practice["unit_price"]
practice["net_sales"] = (
    practice["gross_sales"] * (1 - practice["discount_rate_clean"])
    + practice["shipping_fee"]
)


In [ ]:
# Exercise 1 Solution

# Start with the default value
practice["review_needed"] = "No"

# Change selected rows
practice.loc[
    practice["status"].isin(["Returned", "Cancelled"]),
    "review_needed"
] = "Yes"

practice[["order_id", "status", "review_needed"]].head(10)


In [ ]:
# Exercise 2 Solution

# Start with 0 for all rows
practice["high_shipping_flag"] = 0

# Change to 1 only when shipping_fee is greater than 10
practice.loc[
    practice["shipping_fee"] > 10,
    "high_shipping_flag"
] = 1

practice[["order_id", "shipping_fee", "high_shipping_flag"]].head(10)


In [ ]:
# Exercise 3 Solution

# Default group
practice["discount_group"] = "No Discount"

# Small discount: greater than 0 and up to 10%
practice.loc[
    (practice["discount_rate_clean"] > 0)
    & (practice["discount_rate_clean"] <= 0.10),
    "discount_group"
] = "Small Discount"

# Large discount: more than 10%
practice.loc[
    practice["discount_rate_clean"] > 0.10,
    "discount_group"
] = "Large Discount"

practice[["order_id", "discount_rate_clean", "discount_group"]].head(15)


In [ ]:
# Exercise 4 Solution

practice.loc[
    practice["status"].isin(["Returned", "Cancelled"]),
    "net_sales"
] = 0

practice.loc[
    practice["status"].isin(["Returned", "Cancelled"]),
    ["order_id", "status", "net_sales"]
].head(10)


In [ ]:
# Exercise 5 Solution

practice["important_order"] = "No"

practice.loc[
    (practice["net_sales"] >= 300)
    & (practice["channel"] == "Online"),
    "important_order"
] = "Yes"

practice[["order_id", "channel", "net_sales", "important_order"]].head(15)


In [ ]:
# Exercise 6 Solution

# Copy the original shipping fee into a new column
practice["shipping_fee_adjusted"] = practice["shipping_fee"]

# Remove shipping fee for returned or cancelled orders
practice.loc[
    practice["status"].isin(["Returned", "Cancelled"]),
    "shipping_fee_adjusted"
] = 0

practice[["order_id", "status", "shipping_fee", "shipping_fee_adjusted"]].head(15)


In [ ]:
# Exercise 7 Solution

practice.loc[
    practice["country"].isna(),
    "country"
] = "Unknown"

practice.loc[
    practice["customer_segment"].isna(),
    "customer_segment"
] = "Unassigned"

practice[["country", "customer_segment"]].isna().sum()


In [ ]:
# Exercise 8 Solution

# Start with the default label
practice["sales_note"] = "Regular order"

# High value completed order
practice.loc[
    (practice["net_sales"] >= 300)
    & (practice["status"] == "Completed"),
    "sales_note"
] = "High value completed order"

# Problem order
practice.loc[
    practice["status"].isin(["Returned", "Cancelled"]),
    "sales_note"
] = "Problem order"

practice[["order_id", "status", "net_sales", "sales_note"]].head(15)


In [ ]:
# Exercise 9 Solution

practice.loc[
    practice["status"] == "Cancelled",
    ["net_sales", "sales_note"]
] = [0, "Cancelled order"]

practice.loc[
    practice["status"] == "Cancelled",
    ["order_id", "status", "net_sales", "sales_note"]
].head(10)


In [ ]:
# Exercise 10 Solution

channel_dummies = pd.get_dummies(
    practice["channel"],
    prefix="channel",
    dummy_na=True
)

practice_with_dummies = pd.concat(
    [practice, channel_dummies],
    axis=1
)

practice_with_dummies.filter(regex="^channel_").head()


## Wrap-up: when should beginners use `.loc`?

Use `.loc` when you want to:

- View rows that match a condition
- Update values in one column
- Update values in multiple columns
- Create a default column and then revise selected rows
- Avoid chained assignment problems

The most important pattern to remember is:

```python
df.loc[row_condition, column_name] = new_value
```
